# 03 - Finetune YOLOPv2 (Colab GPU)

This notebook performs mixed-domain finetuning of YOLOPv2 using synthetic CARLA labels + real pseudo labels,
with adversarial domain adaptation through a Gradient Reversal Layer (GRL) discriminator.

Paper references implemented in this notebook:
- Optimal domain adaptive object detection with self-training and adversarial-based approach (arXiv 2020).
- Pseudo-labeling for scalable 3D object detection (Caine et al., arXiv 2021).


## Section 1 - Setup
Install dependencies, clone official YOLOPv2, mount Google Drive, and define paths.


In [ ]:
%pip install -q onnx onnxruntime onnxruntime-tools onnxruntime-gpu torchvision albumentations pandas opencv-python matplotlib tqdm

import os
import sys
from pathlib import Path

if not Path('/content/YOLOPv2').exists():
    !git clone https://github.com/CAIC-AD/YOLOPv2 /content/YOLOPv2

from google.colab import drive
drive.mount('/content/drive')

REPO_ROOT = Path('/content/YOLOPv2')
PROJECT_ROOT = Path('/content/drive/MyDrive/sdcar-perception')
CARLA_MANIFEST = PROJECT_ROOT / 'data' / 'carla' / 'manifest.csv'
PSEUDO_MANIFEST = PROJECT_ROOT / 'data' / 'pseudo' / 'manifest.csv'
CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints'
EXPORT_DIR = PROJECT_ROOT / 'exports'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(REPO_ROOT))
sys.path.append(str(PROJECT_ROOT))

print('Repo:', REPO_ROOT)
print('Project:', PROJECT_ROOT)


## Section 2 - Dataset class
Build a mixed-domain PyTorch Dataset from both manifests with augmentations and required outputs.


In [ ]:
import random
import numpy as np
import pandas as pd
import cv2
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.ops import box_convert
from torchvision import transforms

class MixedPerceptionDataset(Dataset):
    def __init__(self, manifests, image_size=640, augment=True):
        frames = []
        for m in manifests:
            df = pd.read_csv(m)
            frames.append(df)
        self.df = pd.concat(frames, ignore_index=True)
        self.image_size = image_size
        self.augment = augment
        self.color_jitter = transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05)

    def __len__(self):
        return len(self.df)

    def _read_yolo_boxes(self, path):
        boxes = []
        p = Path(path)
        if not p.exists():
            return torch.zeros((0, 5), dtype=torch.float32)
        for line in p.read_text(encoding='utf-8').splitlines():
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls_id, cx, cy, w, h = parts
            boxes.append([float(cx), float(cy), float(w), float(h), float(cls_id)])
        if not boxes:
            return torch.zeros((0, 5), dtype=torch.float32)
        return torch.tensor(boxes, dtype=torch.float32)

    def _augment(self, image, lane_mask, drivable_mask, boxes):
        h, w = image.shape[:2]

        if self.augment and random.random() < 0.5:
            image = cv2.flip(image, 1)
            lane_mask = cv2.flip(lane_mask, 1)
            drivable_mask = cv2.flip(drivable_mask, 1)
            if boxes.shape[0] > 0:
                boxes[:, 0] = 1.0 - boxes[:, 0]

        if self.augment:
            image = np.array(self.color_jitter(transforms.ToPILImage()(image)))

        if self.augment and random.random() < 0.5:
            crop_ratio = random.uniform(0.85, 1.0)
            new_w, new_h = int(w * crop_ratio), int(h * crop_ratio)
            x0 = random.randint(0, w - new_w)
            y0 = random.randint(0, h - new_h)
            image = image[y0:y0+new_h, x0:x0+new_w]
            lane_mask = lane_mask[y0:y0+new_h, x0:x0+new_w]
            drivable_mask = drivable_mask[y0:y0+new_h, x0:x0+new_w]

        image = cv2.resize(image, (self.image_size, self.image_size), interpolation=cv2.INTER_LINEAR)
        lane_mask = cv2.resize(lane_mask, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST)
        drivable_mask = cv2.resize(drivable_mask, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST)

        return image, lane_mask, drivable_mask, boxes

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(str(row['image_path']))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        lane_mask = cv2.imread(str(row['lane_mask_path']), cv2.IMREAD_GRAYSCALE)
        drivable_mask = cv2.imread(str(row['drivable_mask_path']), cv2.IMREAD_GRAYSCALE)

        boxes = self._read_yolo_boxes(row['label_path'])
        image, lane_mask, drivable_mask, boxes = self._augment(image, lane_mask, drivable_mask, boxes)

        image_t = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        lane_t = torch.from_numpy((lane_mask > 0).astype(np.float32)).unsqueeze(0)
        drivable_t = torch.from_numpy((drivable_mask > 0).astype(np.float32)).unsqueeze(0)
        domain_t = torch.tensor([float(row['domain'])], dtype=torch.float32)

        return {
            'image': image_t,
            'boxes': boxes,
            'lane_mask': lane_t,
            'drivable_mask': drivable_t,
            'domain_label': domain_t,
        }


def collate_fn(batch):
    return {
        'image': torch.stack([b['image'] for b in batch], dim=0),
        'boxes': [b['boxes'] for b in batch],
        'lane_mask': torch.stack([b['lane_mask'] for b in batch], dim=0),
        'drivable_mask': torch.stack([b['drivable_mask'] for b in batch], dim=0),
        'domain_label': torch.stack([b['domain_label'] for b in batch], dim=0),
    }

full_ds = MixedPerceptionDataset([CARLA_MANIFEST, PSEUDO_MANIFEST], image_size=640, augment=True)
val_size = max(1, int(0.1 * len(full_ds)))
train_size = len(full_ds) - val_size
train_ds, val_ds = random_split(full_ds, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2, pin_memory=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_fn)

print('Train:', len(train_ds), 'Val:', len(val_ds))


## Section 3 - Model assembly
Load official YOLOPv2 checkpoint, attach domain discriminator on P4, and freeze backbone layers 0-50 for 5 epochs.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

# Official YOLOPv2 internals (paths may vary with upstream commits).
from models.yolo import Model
from utils.loss import ComputeLoss

from src.academic.discriminator import DomainDiscriminator

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Download checkpoint from CAIC-AD backup on HuggingFace if absent.
ckpt_path = CHECKPOINT_DIR / 'yolopv2_official.pt'
if not ckpt_path.exists():
    !wget -O {ckpt_path} https://huggingface.co/nicehuster/yolopv2/resolve/main/yolopv2.pt

ckpt = torch.load(ckpt_path, map_location='cpu')
yolo = Model(cfg=REPO_ROOT / 'cfg' / 'yolopv2.yaml', ch=3, nc=1)
yolo.load_state_dict(ckpt['state_dict'] if 'state_dict' in ckpt else ckpt, strict=False)
yolo = yolo.to(device)

discriminator = DomainDiscriminator().to(device)
compute_loss = ComputeLoss(yolo)

# Freeze backbone layers 0-50 for first 5 epochs.
for i, (name, p) in enumerate(yolo.named_parameters()):
    if i <= 50:
        p.requires_grad = False

params = [p for p in yolo.parameters() if p.requires_grad] + list(discriminator.parameters())
optimizer = torch.optim.AdamW(params, lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

print('Model + discriminator ready on', device)


## Section 4 - Training loop (20 epochs)
Train heads + discriminator for epochs 1-5, then unfreeze all layers for end-to-end training through epoch 20.


In [ ]:
import math

num_epochs = 20
best_loss = math.inf
best_ckpt = CHECKPOINT_DIR / 'yolopv2_best.pt'

for epoch in range(1, num_epochs + 1):
    yolo.train()
    discriminator.train()

    if epoch == 6:
        for p in yolo.parameters():
            p.requires_grad = True
        optimizer = torch.optim.AdamW(list(yolo.parameters()) + list(discriminator.parameters()), lr=1e-4, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=(num_epochs - 5))

    running = {'total': 0.0, 'det': 0.0, 'lane': 0.0, 'drivable': 0.0, 'domain': 0.0}

    for step, batch in enumerate(tqdm(train_loader, desc=f'Epoch {epoch}'), start=1):
        image = batch['image'].to(device)
        lane_gt = batch['lane_mask'].to(device)
        drivable_gt = batch['drivable_mask'].to(device)
        domain_gt = batch['domain_label'].to(device)

        optimizer.zero_grad(set_to_none=True)

        # NOTE: Adapt unpacking to upstream YOLOPv2 forward signature if needed.
        preds = yolo(image)
        det_pred, lane_pred, drivable_pred, p4_feat = preds['det'], preds['lane'], preds['drivable'], preds['p4']

        targets = batch['boxes']
        l_det, _ = compute_loss(det_pred, targets)
        l_lane = F.binary_cross_entropy(lane_pred, lane_gt)
        l_drivable = F.binary_cross_entropy(drivable_pred, drivable_gt)

        # Domain adaptation from arXiv 2020 via GRL + discriminator.
        disc_out = discriminator(p4_feat)
        l_domain = F.binary_cross_entropy(disc_out, domain_gt)

        l_total = l_det + l_lane + l_drivable + 0.1 * l_domain
        l_total.backward()
        optimizer.step()

        running['total'] += float(l_total.item())
        running['det'] += float(l_det.item())
        running['lane'] += float(l_lane.item())
        running['drivable'] += float(l_drivable.item())
        running['domain'] += float(l_domain.item())

        if step % 10 == 0:
            print(
                f"epoch={epoch} step={step} "
                f"L_total={running['total']/step:.4f} "
                f"L_det={running['det']/step:.4f} "
                f"L_lane={running['lane']/step:.4f} "
                f"L_drivable={running['drivable']/step:.4f} "
                f"L_domain={running['domain']/step:.4f}"
            )

    scheduler.step()

    epoch_loss = running['total'] / max(len(train_loader), 1)
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save({'yolo': yolo.state_dict(), 'disc': discriminator.state_dict(), 'epoch': epoch}, best_ckpt)

    if epoch % 5 == 0:
        torch.save({'yolo': yolo.state_dict(), 'disc': discriminator.state_dict(), 'epoch': epoch}, CHECKPOINT_DIR / f'yolopv2_epoch_{epoch}.pt')

print('Training done. Best loss:', best_loss)


## Section 5 - Export
Load best checkpoint, remove discriminator branch, export clean YOLOPv2 ONNX, and apply INT8 quantization.


In [ ]:
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType

best = torch.load(best_ckpt, map_location=device)
yolo.load_state_dict(best['yolo'], strict=False)
yolo.eval()

# Discriminator is excluded from final export graph.
dummy = torch.randn(1, 3, 640, 640, device=device)
onnx_fp32 = EXPORT_DIR / 'yolopv2_finetuned.onnx'
onnx_int8 = EXPORT_DIR / 'yolopv2_finetuned_int8.onnx'

torch.onnx.export(
    yolo,
    dummy,
    str(onnx_fp32),
    opset_version=16,
    input_names=['images'],
    output_names=['det', 'lane', 'drivable'],
    do_constant_folding=True,
    dynamic_axes={'images': {0: 'batch'}, 'det': {0: 'batch'}, 'lane': {0: 'batch'}, 'drivable': {0: 'batch'}},
    dynamo=False,
)

quantize_dynamic(
    model_input=str(onnx_fp32),
    model_output=str(onnx_int8),
    weight_type=QuantType.QInt8,
)

print('Exported:', onnx_int8)


## Section 6 - Quick validation
Run ONNX inference on 10 CARLA + 10 dashcam frames, visualize outputs, and report CARLA mAP on held-out split.


In [ ]:
import matplotlib.pyplot as plt
import onnxruntime as ort

session = ort.InferenceSession(str(onnx_int8), providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])

carla_images = sorted((PROJECT_ROOT / 'data' / 'carla' / 'images').glob('*.jpg'))[:10]
dashcam_images = sorted((PROJECT_ROOT / 'data' / 'pseudo' / 'images').glob('*.jpg'))[:10]
samples = carla_images + dashcam_images

def run_onnx(image_path):
    img = cv2.imread(str(image_path))
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    x = cv2.resize(rgb, (640, 640)).astype(np.float32) / 255.0
    x = np.transpose(x, (2, 0, 1))[None, ...]
    out = session.run(None, {'images': x})
    return rgb, out

for p in samples:
    rgb, out = run_onnx(p)
    det, lane, drivable = out[0], out[1], out[2]

    lane_vis = (lane[0, 0] > 0.5).astype(np.uint8) * 255
    drivable_vis = (drivable[0, 0] > 0.5).astype(np.uint8) * 255

    fig, ax = plt.subplots(1, 3, figsize=(12, 4))
    ax[0].imshow(rgb); ax[0].set_title('Image'); ax[0].axis('off')
    ax[1].imshow(lane_vis, cmap='gray'); ax[1].set_title('Lane mask'); ax[1].axis('off')
    ax[2].imshow(drivable_vis, cmap='gray'); ax[2].set_title('Drivable mask'); ax[2].axis('off')
    plt.tight_layout()
    plt.show()

# Placeholder mAP evaluation hook against held-out CARLA split.
# Replace with official YOLOPv2 evaluation utility if available in current commit.
carla_map = 0.0
print(f'CARLA validation mAP (10% split): {carla_map:.4f}')
